# Sintesi finale - Check-gate 2 su KIMORE ex5

Riepilogo completo di tutti i tentativi fatti per prevedere `cTS` (punteggio clinico) da feature cinematiche dello scheletro Kinect su ex5 (squat), 75 soggetti. Ogni riga qui sotto corrisponde a un run reale di uno script in `src/` (comando riportato), non a un numero ricalcolato: i risultati completi per-fold sono nei log delle run e nei notebook 02-04.

Regola trasversale del playbook rispettata: **onestÃ ** - si riporta anche ciÃ² che non funziona, senza cherry-picking.

In [1]:
import pandas as pd
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 160)

## 1. Regressione continua (cTS), tre famiglie di feature

In [2]:
reg_families = pd.DataFrame([
    {"feature_set": "37 cinematiche (ROM/vel/tempo, ginocchio+anca)", "n_feat": 37,
     "dummy_mae": 0.077, "best_model": "ridge", "best_mae": 0.077, "best_spearman": "0.120+/-0.220",
     "esito": "nessun segnale (pari al dummy, segno instabile)"},
    {"feature_set": "pooling multi-esercizio (stesse feature, tutti i 5 esercizi)", "n_feat": 37,
     "dummy_mae": None, "best_model": "-", "best_mae": None, "best_spearman": "max 2/185 test sig.",
     "esito": "nessun segnale in nessun esercizio (attesi ~9 per caso, trovati 2)"},
    {"feature_set": "traiettoria PCA (10 comp.) + dominio frequenza", "n_feat": 22,
     "dummy_mae": None, "best_model": "-", "best_mae": None, "best_spearman": "0/22 sig.",
     "esito": "nessun segnale, sotto il livello del caso"},
    {"feature_set": "tronco/bacino (flessione, lean laterale, rotazione)", "n_feat": 24,
     "dummy_mae": None, "best_model": "-", "best_mae": None, "best_spearman": "1/24 sig.",
     "esito": "nessun segnale (livello del caso)"},
    {"feature_set": "smoothness/jerk (accelerazione, jerk normalizzato)", "n_feat": 20,
     "dummy_mae": None, "best_model": "-", "best_mae": None, "best_spearman": "0/20 sig.",
     "esito": "nessun segnale"},
    {"feature_set": "combinata 83 (cinematiche+traiettoria+tronco)", "n_feat": 83,
     "dummy_mae": 0.077, "best_model": "rf", "best_mae": 0.080, "best_spearman": "0.057+/-0.237",
     "esito": "nessun segnale, leggermente peggio del dummy"},
])
print(reg_families.to_string(index=False))

                                                 feature_set  n_feat  dummy_mae best_model  best_mae       best_spearman                                                              esito
              37 cinematiche (ROM/vel/tempo, ginocchio+anca)      37      0.077      ridge     0.077       0.120+/-0.220                    nessun segnale (pari al dummy, segno instabile)
pooling multi-esercizio (stesse feature, tutti i 5 esercizi)      37        NaN          -       NaN max 2/185 test sig. nessun segnale in nessun esercizio (attesi ~9 per caso, trovati 2)
              traiettoria PCA (10 comp.) + dominio frequenza      22        NaN          -       NaN           0/22 sig.                          nessun segnale, sotto il livello del caso
         tronco/bacino (flessione, lean laterale, rotazione)      24        NaN          -       NaN           1/24 sig.                                  nessun segnale (livello del caso)
          smoothness/jerk (accelerazione, jerk normalizzato)

## 2. Confronto relativo a coppie (idea CoRe), permutazione

Comando: `notebooks/03_pairwise_relative_check.ipynb`. 2775 coppie di soggetti, differenza di feature vs differenza di cTS, correzione family-wise per 57 feature testate insieme.

In [3]:
pairwise = pd.DataFrame([
    {"metrica": "massimo |rho| osservato (sym_hip)", "valore": 0.247},
    {"metrica": "p-value globale (family-wise, 57 feature)", "valore": 0.4877},
])
print(pairwise.to_string(index=False))
print("\nEsito: nessun segnale relativo, nemmeno a coppie.")

                                  metrica  valore
        massimo |rho| osservato (sym_hip)  0.2470
p-value globale (family-wise, 57 feature)  0.4877

Esito: nessun segnale relativo, nemmeno a coppie.


## 3. Classificazione good/bad (soglia mediana di cTS)

Comando: `python src/quality_model.py --features data/features_ex5_classification.csv --target quality_label --task classification --repeats 10`. Nested CV ripetuta, 5x10=50 split esterni, baseline DummyClassifier.

In [4]:
classif_variants = pd.DataFrame([
    {"variante": "59 feature (cinematiche+traiettoria)", "n_feat": 59,
     "dummy_auc": 0.512, "logreg_auc": 0.599, "rf_auc": 0.622, "mlp_auc": 0.586},
    {"variante": "+ tronco (83 feature)", "n_feat": 83,
     "dummy_auc": 0.512, "logreg_auc": 0.564, "rf_auc": 0.629, "mlp_auc": None},
    {"variante": "+ smoothness (79 feature)", "n_feat": 79,
     "dummy_auc": 0.512, "logreg_auc": 0.588, "rf_auc": 0.630, "mlp_auc": None},
    {"variante": "terzili estremi, n=54 (classi piu separate)", "n_feat": 59,
     "dummy_auc": 0.530, "logreg_auc": 0.557, "rf_auc": 0.505, "mlp_auc": None},
])
print(classif_variants.to_string(index=False))
print("\nOsservazione chiave: l'AUC resta sempre in banda 0.56-0.63 qualunque feature o modello")
print("si aggiunga (plateau, non un segnale che si rafforza), e con classi piu nette (terzili) rf")
print("CROLLA al caso invece di migliorare. Un segnale vero sopravvive a entrambi i cambi: qui no.")

                                   variante  n_feat  dummy_auc  logreg_auc  rf_auc  mlp_auc
       59 feature (cinematiche+traiettoria)      59      0.512       0.599   0.622    0.586
                      + tronco (83 feature)      83      0.512       0.564   0.629      NaN
                  + smoothness (79 feature)      79      0.512       0.588   0.630      NaN
terzili estremi, n=54 (classi piu separate)      59      0.530       0.557   0.505      NaN

Osservazione chiave: l'AUC resta sempre in banda 0.56-0.63 qualunque feature o modello
si aggiunga (plateau, non un segnale che si rafforza), e con classi piu nette (terzili) rf
CROLLA al caso invece di migliorare. Un segnale vero sopravvive a entrambi i cambi: qui no.


## 4. Modello piu flessibile (MLP / rete neurale) sulle 59 feature

Test decisivo: se nemmeno un modello non-lineare piu espressivo di RF/ridge/logreg batte il dummy in modo robusto, l'assenza di segnale non e un limite della scelta di modello.

Comando: `python src/quality_model.py --features data/features_ex5_59.csv --target cTS --task regression --repeats 10` (e l'equivalente --task classification), con MLPRegressor/MLPClassifier aggiunti a `build_models` in `quality_model.py`.

In [5]:
mlp_result = pd.DataFrame([
    {"task": "regression (mae)",  "dummy": 0.077, "ridge/logreg": 0.078, "rf": 0.081, "mlp": 0.113},
    {"task": "regression (spearman)", "dummy": float("nan"), "ridge/logreg": -0.009, "rf": -0.067, "mlp": -0.038},
    {"task": "classification (auc)", "dummy": 0.512, "ridge/logreg": 0.599, "rf": 0.622, "mlp": 0.586},
])
print(mlp_result.to_string(index=False))
print("\nIn regressione l'MLP e nettamente PEGGIORE del dummy (mae 0.113 vs 0.077, R2 medio")
print("-1.56: overfitting severo con 59 feature su 75 soggetti). In classificazione l'MLP resta")
print("nello stesso plateau di logreg/rf, non lo supera. Un modello piu flessibile non recupera")
print("nessun segnale nascosto: il limite non e la scelta di modello.")

                 task  dummy  ridge/logreg     rf    mlp
     regression (mae)  0.077         0.078  0.081  0.113
regression (spearman)    NaN        -0.009 -0.067 -0.038
 classification (auc)  0.512         0.599  0.622  0.586

In regressione l'MLP e nettamente PEGGIORE del dummy (mae 0.113 vs 0.077, R2 medio
-1.56: overfitting severo con 59 feature su 75 soggetti). In classificazione l'MLP resta
nello stesso plateau di logreg/rf, non lo supera. Un modello piu flessibile non recupera
nessun segnale nascosto: il limite non e la scelta di modello.


## Nota metodologica: bug di leakage trovato e corretto

Nel costruire il dataset di regressione con le feature di tronco, `quality_label` (derivato direttamente da `cTS` tramite soglia mediana) era rimasto tra le feature per errore - leakage puro del target, che aveva prodotto uno Spearman fittizio di 0.6-0.9. Trovato ricontrollando i file da zero, verificato che nessun altro dataset avesse lo stesso problema, corretto e rilanciato pulito (risultato incluso nella tabella della sezione 1).

## Verdetto

Nove famiglie/formulazioni indipendenti testate (regressione assoluta x3, relativa a coppie, classificazione x4 varianti, piÃ¹ il test con modello non-lineare). L'unico scostamento dal rumore (classificazione a soglia mediana, AUC~0.6) non sopravvive a due controlli di robustezza indipendenti (aggiunta di feature pertinenti, soglia piu netta). **Check-gate 2 resta rosso su KIMORE ex5**: nessuna combinazione di feature o formulazione del problema produce un modello statisticamente affidabile con questo campione (~75 soggetti) e questo target (punteggio clinico generale, non specifico dell'esecuzione).